In [1]:
%load_ext autoreload
%autoreload 2

In [19]:
import torch
import torch.nn as nn
from tqdm import tqdm
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
from lib.data.datasets import AccRawDataset
from lib.data.dataloading import load_raw, load_nursing_5_class
from lib.config import RAW_DIR
from lib.modules import optimization_loop_xonly,sample_regnet, optimization_loop_multi_class
from lib.models import RegNetv3, RegNetMAEv3, CosineMSELoss, RegNetv3Ci
from pathlib import Path
import json

In [3]:
CONFIG = {
    'WINDOW_SIZE':2001,
    'WINDOW_STRIDE':2001 // 16,
    'NURSING_STRIDE': 2001 // 16,
    'BATCH_SIZE': 512,
    'LEARNING_RATE': 1e-3,
    'CLASS_LR': 3e-4,
    'ENC_LEARNING_RATE': 5e-5,
    'TEST_SIZE': 0.1,
    'NURSING_TEST_SIZE': 0.25,
    'DEVICE': 'cuda:1',
    'DEPTHI': [2],
    'WIDTHI': [64],
    'NTL': 1,
    'DMODEL': 64,
    'MASKPCT': 0.5,
    'PDROPOUT': 0.0
}

In [4]:
trainloader, testloader = load_raw(
    RAW_DIR,
    CONFIG['WINDOW_SIZE'],
    test_size=CONFIG['TEST_SIZE'],
    batch_size=CONFIG['BATCH_SIZE'],
    shuffle_test=True,
    chunk_len_hrs=0.25,
    stride=CONFIG['WINDOW_STRIDE']
)

Using all available sessions
Using Directories: ['2022-12-20_15_10_43.pt', '10-28_13_18_42.pt', '2022-12-10_14_27_45.pt', '2023-11-11_17_50_20.pt', '2022-12-20_11_18_54.pt', '2022-12-08_14_30_54.pt', '2023-11-01_15_49_48.pt', '2022-12-20_10_54_30.pt', '2023-10-26_15_32_20.pt', '2022-12-12_07_03_40.pt', '11-07_17_43_30.pt', '2023-02-11_13_51_34.pt', '11-07_12_58_43.pt', '2024-02-27_09_39_32.pt', '2023-11-02_13_55_22.pt', '2023-11-10_13_11_41.pt', '10-27_00_21_25.pt', '2022-12-21_14_16_02.pt', '2022-12-10_12_57_32.pt', '2018-01-01_13_33_47.pt', '11-07_15_03_24.pt', '11-07_17_29_01.pt', '11-08_08_27_30.pt', '2022-12-19_08_37_36.pt', '2022-12-06_17_27_56.pt', '2022-12-06_15_48_49.pt', '11-01_20_34_28.pt', '2022-12-06_18_21_08.pt', '10-27_09_45_42.pt', '2023-11-01_15_47_52.pt', '2024-02-22_18_21_48.pt', '2022-12-08_11_52_09.pt', '2018-01-03_13_28_35.pt', '11-08_07_17_47.pt', '2022-12-08_19_04_00.pt', '11-02_19_28_19.pt', '11-10_08_54_24.pt', '10-27_00_20_15.pt', '11-01_20_54_52.pt', '11-07_

In [7]:
CONFIG = {
    'WINDOW_SIZE':2001,
    'WINDOW_STRIDE':2001,
    'NURSING_STRIDE': 2001,
    'BATCH_SIZE':512,
    'LEARNING_RATE':3e-4,
    'TEST_SIZE':0.1,
    'NURSING_TEST_SIZE': 0.25,
    'DEVICE':'cuda:0',
}
trainloader, testloader = load_raw(
    RAW_DIR,
    CONFIG['WINDOW_SIZE'],
    test_size=CONFIG['TEST_SIZE'],
    batch_size=CONFIG['BATCH_SIZE'],
    shuffle_test=True,
    chunk_len_hrs=0.25,
    stride=CONFIG['WINDOW_STRIDE']
)

class RegNetMAEv4(nn.Module):
    def __init__(self):
        super().__init__()
        seq_len = 2001
        self.e = nn.Sequential(
            nn.Conv1d(3, 64, 3, padding=1, stride=1),
            nn.LayerNorm(seq_len, elementwise_affine=False),
            nn.ReLU(),
            nn.Conv1d(64, 64, 3, padding=1, stride=1),
            nn.LayerNorm(seq_len, elementwise_affine=False),
            nn.ReLU(),
            nn.Conv1d(64, 3, 3, padding=1, stride=1),
        )
    def forward(self, x):
        return self.e(x)
    
model = RegNetMAEv4().to(CONFIG['DEVICE'])
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['LEARNING_RATE'])

Using all available sessions
Using Directories: ['2022-12-20_15_10_43.pt', '10-28_13_18_42.pt', '2022-12-10_14_27_45.pt', '2023-11-11_17_50_20.pt', '2022-12-20_11_18_54.pt', '2022-12-08_14_30_54.pt', '2023-11-01_15_49_48.pt', '2022-12-20_10_54_30.pt', '2023-10-26_15_32_20.pt', '2022-12-12_07_03_40.pt', '11-07_17_43_30.pt', '2023-02-11_13_51_34.pt', '11-07_12_58_43.pt', '2024-02-27_09_39_32.pt', '2023-11-02_13_55_22.pt', '2023-11-10_13_11_41.pt', '10-27_00_21_25.pt', '2022-12-21_14_16_02.pt', '2022-12-10_12_57_32.pt', '2018-01-01_13_33_47.pt', '11-07_15_03_24.pt', '11-07_17_29_01.pt', '11-08_08_27_30.pt', '2022-12-19_08_37_36.pt', '2022-12-06_17_27_56.pt', '2022-12-06_15_48_49.pt', '11-01_20_34_28.pt', '2022-12-06_18_21_08.pt', '10-27_09_45_42.pt', '2023-11-01_15_47_52.pt', '2024-02-22_18_21_48.pt', '2022-12-08_11_52_09.pt', '2018-01-03_13_28_35.pt', '11-08_07_17_47.pt', '2022-12-08_19_04_00.pt', '11-02_19_28_19.pt', '11-10_08_54_24.pt', '10-27_00_20_15.pt', '11-01_20_54_52.pt', '11-07_

In [6]:
# while True:
#     d,w,_ = sample_regnet()
#     CONFIG['DEPTHI'] = d
#     CONFIG['WIDTHI'] = w
#     model = RegNetMAEv3(CONFIG=CONFIG).to(CONFIG['DEVICE'])
#     print(d,w,sum([p.numel() for p in model.parameters()]))
#     if sum([p.numel() for p in model.parameters()]) < 4000000:
#         break
class NormalizedRegnetMAEv3(nn.Module):
    def __init__(self, CONFIG):
        super().__init__()
        self.norm = nn.LayerNorm((3, CONFIG['WINDOW_SIZE']), elementwise_affine=False)
        self.model = RegNetMAEv3(CONFIG=CONFIG)
    def forward(self, x):
        return self.model(self.norm(x))
model = NormalizedRegnetMAEv3(CONFIG=CONFIG).to(CONFIG['DEVICE'])
class NormalizedCosineMSELoss(nn.Module):
    def __init__(self, window_size):
        super().__init__()
        self.norm = nn.LayerNorm((3, window_size), elementwise_affine=False)
        self.criterion = CosineMSELoss()
    def forward(self, x, y):
        return self.criterion(x, self.norm(y))

criterion = NormalizedCosineMSELoss(CONFIG['WINDOW_SIZE'])
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
sum([p.numel() for p in model.parameters()])

latent dim: 500


98776

In [17]:
outdir = f'dev/9_regnet-mae/prototyping-2-27-24/normalized/first-try'
optimization_loop_xonly(
    model,
    trainloader,
    testloader,
    criterion,
    optimizer,
    epochs=3500,
    patience=20,
    config=CONFIG,
    continue_training=False,
    device=CONFIG['DEVICE'],
    outdir=outdir,
    writer=outdir,
    label=f'MAE {CONFIG["MASKPCT"]}%-{CONFIG["DMODEL"]}: '
)

  0%|          | 0/3500 [00:00<?, ?it/s]

MAE 0.5%-64: : Epoch 24: Train Loss: 0.066576: Dev Loss: 0.055745:   1%|          | 25/3500 [46:46<108:22:15, 112.27s/it]


KeyboardInterrupt: 

In [17]:
outdir = Path('/home/musa/eating-detection/dev/9_regnet-mae/prototyping-2-27-24/normalized/first-try')
CONFIG = json.load(open(outdir / 'config.json'))

In [28]:
from collections import OrderedDict
state = torch.load(outdir / 'best_model.pt')
state = OrderedDict({k.replace('model.',''):v for k, v in state.items()})
torch.save(state, outdir / 'best_model_fixedkeys.pt')

In [32]:
from lib.models import RegNetv3
from lib.modules import optimization_loop_multi_class
from lib.data.dataloading import load_nursing_5_class

CONFIG['NURSING_STRIDE'] = CONFIG['WINDOW_SIZE']
CONFIG['FROZEN'] = False
CONFIG['BATCH_SIZE'] = 512
CONFIG['WEIGHTS_FILE'] = f'{outdir}/best_model_fixedkeys.pt' 
CONFIG['PRETRAINED'] = bool(CONFIG['WEIGHTS_FILE'])

nursing_trainloader, nursing_testloader = load_nursing_5_class(
    range(11,71), 
    CONFIG['WINDOW_SIZE'], 
    test_size=CONFIG['NURSING_TEST_SIZE'], 
    batch_size=CONFIG['BATCH_SIZE'],
    stride=CONFIG['NURSING_STRIDE'],
)
class NormalizedRegNetv3(nn.Module):
    def __init__(self, CONFIG):
        super().__init__()
        self.norm = nn.LayerNorm((3, CONFIG['WINDOW_SIZE']), elementwise_affine=False)
        self.model = RegNetv3(CONFIG=CONFIG)
    def forward(self, x):
        return self.model(self.norm(x))
model = NormalizedRegNetv3(CONFIG=CONFIG).to(CONFIG['DEVICE'])
criterion = nn.CrossEntropyLoss()
# optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
optimizer = torch.optim.Adam(
    [
        {"params": model.model.o.parameters()},
        {"params": [
            *model.model.e.parameters(),
            # *model.model.trans_skip.parameters(), 
            # *model.model.transformer_encoder.parameters()
        ], "lr": (lr:=5e-5)},
    ],
    lr=3e-4
)

noutdir = Path('dev/null/pretrain-norm-layerwiselr')
optimization_loop_multi_class(
    model,
    nursing_trainloader,
    nursing_testloader,
    criterion,
    optimizer,
    epochs=500,
    device=CONFIG['DEVICE'],
    patience=50,
    outdir=noutdir,
    writer=noutdir,
    config=CONFIG
)

Model is loading pretrained encoder


: Epoch 50: Train Loss: 1.4149: Dev Loss: 1.4312, Dev F1: 0.11161:  10%|█         | 50/500 [00:25<03:53,  1.93it/s]

Early stopping at epoch 50


# Train on Ci

In [ ]:
from lib.models.regnetv3.regnet_encoder import RegNetEncoder
class RegNetv3Ci(nn.Module):
    def __init__(
        self, 
        winsize=None, in_channels=3, stem_out_c=None, d=None, w=None, 
        g=1, p_dropout=0, 
        d_model=64, ntrans=1, nhead=2, trans_dropout=0.01, tran_linear_dim=None,
        weights_file=None, freeze=False,
        CONFIG=None
    ):
        super().__init__()
        if CONFIG:
            winsize = CONFIG['WINDOW_SIZE']
            stem_out_c = CONFIG['WIDTHI'][0]
            d = CONFIG['DEPTHI']
            w = CONFIG['WIDTHI']
            d_model = CONFIG['DMODEL']
            ntrans = CONFIG['NTL']
            p_dropout = CONFIG['PDROPOUT']
            weights_file = CONFIG.get('WEIGHTS_FILE', None)
            freeze = CONFIG.get('FROZEN', False)
        else:
            CONFIG = {
                'WINDOW_SIZE': winsize,
                'MASK_PCT': 0.0,
                'WIDTHI': w,
                'DEPTHI': d,
                'DMODEL': d_model,
                'NTL': ntrans,
                'PDROPOUT': p_dropout,
            }
        if not stem_out_c:
            stem_out_c = w[0]
        
        mae = RegNetMAEv3(CONFIG=CONFIG, tran_linear_dim=2048)
        if weights_file:
            mae.load_state_dict(torch.load(weights_file))
        if freeze:
            for param in mae.parameters():
                param.requires_grad = False
        
        self.e = mae.e
        self.trans_skip = mae.trans_skip
        self.transformer_encoder = mae.transformer_encoder
        # for param in self.trans_skip.parameters():
        #     param.requires_grad = False
        # for param in self.transformer_encoder.parameters():
        #     param.requires_grad = False
        self.o = nn.Sequential(
            nn.AvgPool1d(kernel_size=self.e.latent_dim), # Nxdims[-1]x1
            nn.Flatten(start_dim=1), # Nxdims[-1]
            nn.Linear(in_features=d_model, out_features=5)
        )
    def forward(self, x):
        x = self.e(x)
        x = self.trans_skip(x) + self.transformer_encoder(x)
        x = self.o(x)
        return x

In [ ]:
ae_dir = Path(f'/home/musa/eating-detection/dev/9_regnet-mae/random-search/[4, 13, 3]-[48, 120, 304]')
CONFIG = json.load(open(ae_dir/'config.json'))
CONFIG['WEIGHTS_FILE'] = f'{ae_dir}/best_model.pt'
CONFIG['FROZEN'] = False

model = RegNetv3Ci(CONFIG=CONFIG).to(CONFIG['DEVICE'])
# model = RegNetv3(CONFIG=CONFIG).to(CONFIG['DEVICE'])
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    [
        {"params": model.o.parameters()},
        {"params": [
            *model.e.parameters(),
            # *model.trans_skip.parameters(), 
            # *model.transformer_encoder.parameters()
        ], "lr": (lr:=4.5e-5)},
    ],
    lr=3e-4
)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

In [ ]:
nursing_trainloader, nursing_testloader = load_nursing_5_class(
    range(11,71), 
    CONFIG['WINDOW_SIZE'], 
    test_size=CONFIG['NURSING_TEST_SIZE'], 
    batch_size=CONFIG['BATCH_SIZE'],
    stride=CONFIG['NURSING_STRIDE'],
)

optimization_loop_multi_class(
    model,
    nursing_trainloader,
    nursing_testloader,
    criterion,
    optimizer,
    epochs=1500,
    device=CONFIG['DEVICE'],
    patience=50,
    outdir=(outdirnursing:=f'dev/ci/non-ci-pretrain23'),
    writer=outdirnursing,
    config=CONFIG
)

# Evaluate

In [ ]:
ae_dir = Path(f'/home/musa/eating-detection/dev/9_regnet-mae/random-search-2/mae/[1, 3, 6, 10]-[48, 96, 224, 496]')

CONFIG = json.load((ae_dir / 'config.json').open())
CONFIG['DEVICE'] = 'cuda:1'
info = json.load((ae_dir / 'info.json').open())
model = RegNetMAEv3(CONFIG=CONFIG).to(CONFIG['DEVICE'])
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
model.load_state_dict(torch.load(ae_dir / 'best_model.pt'))

In [22]:
trainloader, testloader = load_raw(
    RAW_DIR,
    CONFIG['WINDOW_SIZE'],
    test_size=CONFIG['TEST_SIZE'],
    batch_size=CONFIG['BATCH_SIZE'],
    shuffle_test=True,
    chunk_len_hrs=0.25,
    stride=CONFIG['WINDOW_STRIDE']
)

model.eval()
with torch.no_grad():
    for X in testloader:
        logits = model(X.to(CONFIG['DEVICE'])).detach().cpu()
        X = nn.LayerNorm((3, CONFIG['WINDOW_SIZE']), elementwise_affine=False)(X)
        # x = model.e(X.to(CONFIG['DEVICE']))
        # encoded = x.detach().cpu()
        # # x = model.mask(x)
        # m = x.detach().cpu()
        # trans_out = model.transformer_encoder(x)
        # x = model.decoder(x + trans_out)
        # trans_out = trans_out.detach().cpu()
        # logits = x.detach().cpu()
        break

Using all available sessions
Using Directories: ['2022-12-20_15_10_43.pt', '10-28_13_18_42.pt', '2022-12-10_14_27_45.pt', '2023-11-11_17_50_20.pt', '2022-12-20_11_18_54.pt', '2022-12-08_14_30_54.pt', '2023-11-01_15_49_48.pt', '2022-12-20_10_54_30.pt', '2023-10-26_15_32_20.pt', '2022-12-12_07_03_40.pt', '11-07_17_43_30.pt', '2023-02-11_13_51_34.pt', '11-07_12_58_43.pt', '2024-02-27_09_39_32.pt', '2023-11-02_13_55_22.pt', '2023-11-10_13_11_41.pt', '10-27_00_21_25.pt', '2022-12-21_14_16_02.pt', '2022-12-10_12_57_32.pt', '2018-01-01_13_33_47.pt', '11-07_15_03_24.pt', '11-07_17_29_01.pt', '11-08_08_27_30.pt', '2022-12-19_08_37_36.pt', '2022-12-06_17_27_56.pt', '2022-12-06_15_48_49.pt', '11-01_20_34_28.pt', '2022-12-06_18_21_08.pt', '10-27_09_45_42.pt', '2023-11-01_15_47_52.pt', '2024-02-22_18_21_48.pt', '2022-12-08_11_52_09.pt', '2018-01-03_13_28_35.pt', '11-08_07_17_47.pt', '2022-12-08_19_04_00.pt', '11-02_19_28_19.pt', '11-10_08_54_24.pt', '10-27_00_20_15.pt', '11-01_20_54_52.pt', '11-07_

In [23]:
i = slice(0,10)
df = pd.DataFrame({
    'x': X[i,0].flatten(), 
    'y': X[i,1].flatten(), 
    'z': X[i,2].flatten(), 
    'x_pred': logits[i,0].flatten(), 
    'y_pred': logits[i,1].flatten(), 
    'z_pred': logits[i,2].flatten()
})
fig = px.line(df, x=df.index, y=['x', 'x_pred', 'y', 'y_pred', 'z', 'z_pred'])
fig.show(renderer='browser')

In [ ]:
i=0
plt.plot(X[i,0])
plt.plot(X[i,1])
plt.plot(X[i,2])
plt.plot(logits[i,0])
plt.plot(logits[i,1])
plt.plot(logits[i,2])

In [ ]:
for j in range(0,len(encoded[i])):
    plt.plot(encoded[i,j])

In [ ]:
for j in range(0,len(m[i])):
    plt.plot(m[i,j])

In [ ]:
for j in range(0,len(trans_out[i])):
    plt.plot(trans_out[i,j])

# Nursing

In [ ]:
from lib.data.dataloading import load_nursing_5_class

nursing_trainloader, nursing_testloader = load_nursing_5_class(
    range(11,71), 
    CONFIG['WINDOW_SIZE'], 
    test_size=1, 
    batch_size=CONFIG['BATCH_SIZE'],
    stride=CONFIG['WINDOW_SIZE'],
)

In [ ]:
embedding = []
Xs = []
ys = []
embedding_c = []
for i,(X,y) in enumerate(tqdm(nursing_testloader)):
    with torch.no_grad():
        Xs.append(X)
        ys.append(y)
        X = X.view(-1, 3, CONFIG['WINDOW_SIZE'])
        X = X.to(CONFIG['DEVICE'])
        x = model.e(X)
        embedding.append(x.detach().cpu().mean(dim=2))
        x = model.transformer_encoder(x)
        embedding_c.append(x.detach().cpu().mean(dim=2))
    
embedding = torch.cat(embedding, dim=0)
Xs = torch.cat(Xs, dim=0)
ys = torch.cat(ys, dim=0)
embedding_c = torch.cat(embedding_c, dim=0)

In [ ]:
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

emb_np = embedding.numpy()

# Applying t-SNE to reduce the dimensions to 2
tsne = TSNE(n_components=2)
data_tsne = tsne.fit_transform(emb_np)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(data_tsne[:, 0], data_tsne[:, 1], marker='o', c=ys)#, c=cluster_labels, cmap='viridis')
plt.title(f't-SNE Visualization of Nursing Embedding (ti) [Maskpct: {CONFIG["MASKPCT"]}, dmodel: {CONFIG["DMODEL"]}]')
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.legend(*scatter.legend_elements(), loc="lower left", title="Classes")
plt.show()

# 'NONE':0,
# 'Eating':1,
# 'Exercise':2,
# 'Medication':3,
# 'Smoking':4,

In [ ]:
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

emb_c_np = embedding_c.numpy()

# Applying t-SNE to reduce the dimensions to 2
tsne = TSNE(n_components=2)
data_tsne = tsne.fit_transform(emb_c_np)

plt.figure(figsize=(8, 6))
plt.scatter(data_tsne[:, 0], data_tsne[:, 1], marker='o', c=ys)#, c=cluster_labels, cmap='viridis')
plt.title(f't-SNE Visualization of Nursing After Transformer (ci) [Maskpct: {CONFIG["MASKPCT"]}, dmodel: {CONFIG["DMODEL"]}]')
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.legend(*scatter.legend_elements(), loc="lower left", title="Classes")
plt.show()

In [ ]:
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

Xs_np = Xs.numpy()

# Applying t-SNE to reduce the dimensions to 2
tsne = TSNE(n_components=2)
data_tsne = tsne.fit_transform(Xs_np)

plt.figure(figsize=(8, 6))
plt.scatter(data_tsne[:, 0], data_tsne[:, 1], marker='o', c=ys)#, c=cluster_labels, cmap='viridis')
plt.title('t-SNE Visualization of Accelerometer Data with 2 components')
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.legend(*scatter.legend_elements(), loc="lower left", title="Classes")
plt.show()

### Classifier

In [ ]:
from lib.models import RegNetv3
from lib.modules import optimization_loop_multi_class
from lib.data.dataloading import load_nursing_5_class

CONFIG['DEVICE'] = 'cuda:0'
CONFIG['FROZEN'] = False
CONFIG['PRETRAINED'] = False
# CONFIG['WEIGHTS_FILE'] =  f'/home/musa/eating-detection/dev/9_regnet-mae/prototype-v3_2-21-24/maskpct0.15_[2]-[64]_LossCosineEmbeddingLossPositive/best_model.pt'

nursing_trainloader, nursing_testloader = load_nursing_5_class(
    range(11,71), 
    CONFIG['WINDOW_SIZE'], 
    test_size=CONFIG['TEST_SIZE'], 
    batch_size=CONFIG['BATCH_SIZE'],
    stride=CONFIG['WINDOW_STRIDE'],
)

model = RegNetv3(CONFIG=CONFIG).to(CONFIG['DEVICE'])
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

# outdir = Path(f'9_regnet-mae/prototype-v3_2-21-24-classifiers/maskpct0.15_[2]-[64]_LossCosineEmbeddingLossPositive')
outdir = Path(f'9_regnet-mae/prototype-v3_2-21-24-classifiers/no-pretrain')
optimization_loop_multi_class(
    model,
    nursing_trainloader,
    nursing_testloader,
    criterion,
    optimizer,
    epochs=200,
    device=CONFIG['DEVICE'],
    patience=50,
    outdir=f'dev/{outdir}',
    writer=f'runs/{outdir}',
    config=CONFIG
)

# No labels

In [ ]:
import matplotlib.pyplot as plt

embedding = []
Xs = []
embedding_c = []
for i,X in enumerate(tqdm(testloader)):
    with torch.no_grad():
        Xs.append(X)
        X = X.view(-1, 3, CONFIG['WINDOW_SIZE'])
        X = X.to(CONFIG['DEVICE'])
        x = model.e(X)
        embedding.append(x.detach().cpu().mean(dim=2))
        x = model.transformer_encoder(x)
        embedding_c.append(x.detach().cpu().mean(dim=2))
    
embedding = torch.cat(embedding, dim=0)
Xs = torch.cat(Xs, dim=0)
embedding_c = torch.cat(embedding_c, dim=0)

In [ ]:
i = 1
fig, axs = plt.subplots(3, 1, figsize=(12, 12))
axs[0].plot(embedding[i])
axs[1].plot(embedding_c[i])
axs[2].plot(Xs[i])
plt.show()

# TSNE

In [ ]:
from sklearn.cluster import KMeans

emb_np = embedding.numpy()

wss = []
for num_clusters in range(1,10):
    kmeans = KMeans(n_clusters=num_clusters)
    cluster_labels = kmeans.fit_predict(emb_np)
    wss.append(kmeans.inertia_)

plt.plot(wss)

In [ ]:
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

emb_np = embedding.numpy()

# Cluster each sample (1001 dims) into one of 4 clusters
# num_clusters = 3
# kmeans = KMeans(n_clusters=num_clusters)
# cluster_labels = kmeans.fit_predict(emb_np)

# Applying t-SNE to reduce the dimensions to 2
tsne = TSNE(n_components=2)
data_tsne = tsne.fit_transform(emb_np)

plt.figure(figsize=(8, 6))
plt.scatter(data_tsne[:, 0], data_tsne[:, 1], marker='o', s=10)#, c=cluster_labels, cmap='viridis')
plt.title('t-SNE Visualization of Embedding (ti) with 2 components')
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.show()

In [ ]:
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

embc_np = embedding_c.numpy()

# Cluster each sample (1001 dims) into one of 4 clusters
# num_clusters = 3
# kmeans = KMeans(n_clusters=num_clusters)
# cluster_labels = kmeans.fit_predict(embc_np)

# Applying t-SNE to reduce the dimensions to 2
tsne = TSNE(n_components=2)
data_tsne = tsne.fit_transform(embc_np)

plt.figure(figsize=(8, 6))
plt.scatter(data_tsne[:, 0], data_tsne[:, 1], marker='o', s=10)#, c=cluster_labels, cmap='viridis')
plt.title('t-SNE Visualization of Embedding after reconstruction (ci) with 2 components')
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.show()

In [ ]:
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

Xs_np = Xs.numpy()

# Cluster each sample (1001 dims) into one of 4 clusters
# num_clusters = 3
# kmeans = KMeans(n_clusters=num_clusters)
# cluster_labels = kmeans.fit_predict(Xs_np)

# Applying t-SNE to reduce the dimensions to 2
tsne = TSNE(n_components=2)
data_tsne = tsne.fit_transform(Xs_np)

plt.figure(figsize=(8, 6))
plt.scatter(data_tsne[:, 0], data_tsne[:, 1], marker='o', s=10)#, c=cluster_labels, cmap='viridis')
plt.title('t-SNE Visualization of Accelerometer Data with 2 components')
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.show()

# Many models

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
n_epochs_for_best = []
maskpcts = []
dmodels = []
best_losses = {}
df = pd.DataFrame(columns=['maskpct', 'dmodel', 'best_loss', 'n_epochs_for_best', 'd', 'w', 'nparams'])

for model_dir in Path('/home/musa/eating-detection/dev/9_regnet-mae/random-search').iterdir():
    config = json.load(open(model_dir/'config.json'))
    info = json.load(open(model_dir/'info.json'))
    model = RegNetMAEv3(CONFIG=config)
    nparams = sum([p.numel() for p in model.parameters()])
    # model.load_state_dict(torch.load(model_dir/'best_model.pt'))

    df.loc[len(df)] = {
        'maskpct': config['MASKPCT'],
        'dmodel': config['DMODEL'],
        'best_loss': info['Loss'],
        'n_epochs_for_best': info['Best Model'],
        'd': config['DEPTHI'],
        'w': config['WIDTHI'],
        'nparams': nparams
    }

df = df.sort_values(['dmodel', 'maskpct'])

In [ ]:
df.drop(df[df['n_epochs_for_best'] < 0].index, inplace=True)

In [ ]:
df

In [ ]:
df['dmodellog'] = np.log2(df['dmodel']) - np.log2(df['dmodel']).min()
fig = plt.figure(figsize=(7.5,10))
ax = fig.add_subplot(1,1,1)
for maskpct in df['maskpct'].unique():
    df[df['maskpct']==maskpct].plot(x='dmodellog', y='best_loss', ax=ax, label=f'Mask Percentage: {maskpct}')

ax.set_xlabel('dmodel')
ax.set_ylabel('best_loss')
ax.set_title('Best Loss vs dmodel for different mask percentages')
ax.set_xticks(df['dmodellog'].unique())
ax.set_xticklabels(df['dmodel'].unique())
ax.legend(loc=(1.02,0))

In [ ]:
n_epochs_for_best = []
maskpcts = []
dmodels = []
best_losses = {}

fig = plt.figure(figsize=(5,5))
ax = fig.add_subplot(1,1,1)
for dmodel in df['dmodel'].unique():
    df[df['dmodel']==dmodel].plot(x='maskpct', y='best_loss', ax=ax, label=f'dmodel: {dmodel}')

ax.set_xlabel('Mask Percentage')
ax.set_ylabel('best_loss')
ax.set_title('Mask Percentage vs Best Loss for different dmodels')
ax.set_xticks(df['maskpct'].unique())
ax.set_xticklabels([f'{int(maskpct*100)}%' for maskpct in df['maskpct'].unique()])
ax.legend(loc=(0.3,-0.5))

In [ ]:
plt.scatter(x=df['nparams'], y=df['best_loss'])